In [ ]:
import sys
sys.path.append("E:\\wikiart_project")

from config import (
    set_seed, get_split_indices, get_transforms,
    WikiArtDataset, NUM_EPOCHS, LEARNING_RATE,
    LOSS_WEIGHTS, NUM_STYLES, NUM_GENRES, SAVE_DIR
)

import os, json
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import models
from datasets import load_dataset
from tqdm import tqdm
import matplotlib.pyplot as plt

set_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print(f"GPU: {torch.cuda.get_device_name(0)}")

print("Loading dataset...")
dataset = load_dataset("huggan/wikiart", split="train")
print(f"Total images: {len(dataset)}")

splits = get_split_indices(len(dataset))
print(f"Train: {len(splits['train'])} | Val: {len(splits['val'])} | Test: {len(splits['test'])}")

# ── ResNet-50 settings ──────────────────────────────────
IMAGE_SIZE = 224
BATCH_SIZE = 32
MODEL_NAME = "resnet50"

train_transform, eval_transform = get_transforms(IMAGE_SIZE)

train_dataset = WikiArtDataset(dataset, splits['train'], train_transform)
val_dataset   = WikiArtDataset(dataset, splits['val'],   eval_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

# ── Multi-task ResNet-50 ─────────────────────────────────
class MultiTaskResNet(nn.Module):
    def __init__(self, num_styles=NUM_STYLES, num_genres=NUM_GENRES):
        super().__init__()
        backbone = models.resnet50(weights='IMAGENET1K_V1')
        self.backbone = nn.Sequential(*list(backbone.children())[:-1])
        self.style_head = nn.Linear(2048, num_styles)
        self.genre_head = nn.Linear(2048, num_genres)

    def forward(self, x):
        features = self.backbone(x).flatten(1)
        return self.style_head(features), self.genre_head(features)

model = MultiTaskResNet().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
scaler = torch.cuda.amp.GradScaler()

# ── Checkpoint resume ────────────────────────────────────
CHECKPOINT_PATH = f"{SAVE_DIR}\\{MODEL_NAME}_checkpoint.pth"
start_epoch = 0
train_losses, val_losses, train_accs, val_accs = [], [], [], []

if os.path.exists(CHECKPOINT_PATH):
    print("Found checkpoint, resuming...")
    ckpt = torch.load(CHECKPOINT_PATH)
    model.load_state_dict(ckpt['model_state'])
    optimizer.load_state_dict(ckpt['optimizer_state'])
    scaler.load_state_dict(ckpt['scaler_state'])
    start_epoch = ckpt['epoch'] + 1
    train_losses = ckpt['train_losses']
    val_losses   = ckpt['val_losses']
    train_accs   = ckpt['train_accs']
    val_accs     = ckpt['val_accs']
    print(f"Resumed from epoch {start_epoch}")

for epoch in range(start_epoch, NUM_EPOCHS):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for images, styles, genres in tqdm(train_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS} [Train]"):
        images, styles, genres = images.to(device), styles.to(device), genres.to(device)
        optimizer.zero_grad()
        with torch.cuda.amp.autocast():
            style_out, genre_out = model(images)
            loss = LOSS_WEIGHTS['style']*criterion(style_out, styles) + LOSS_WEIGHTS['genre']*criterion(genre_out, genres)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        running_loss += loss.item()
        correct += (style_out.argmax(1) == styles).sum().item()
        total += styles.size(0)

    train_losses.append(running_loss/len(train_loader))
    train_accs.append(correct/total)

    model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
    with torch.no_grad():
        for images, styles, genres in tqdm(val_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS} [Val]"):
            images, styles, genres = images.to(device), styles.to(device), genres.to(device)
            with torch.cuda.amp.autocast():
                style_out, genre_out = model(images)
                loss = LOSS_WEIGHTS['style']*criterion(style_out, styles) + LOSS_WEIGHTS['genre']*criterion(genre_out, genres)
            val_loss += loss.item()
            val_correct += (style_out.argmax(1) == styles).sum().item()
            val_total += styles.size(0)

    val_losses.append(val_loss/len(val_loader))
    val_accs.append(val_correct/val_total)
    print(f"Epoch {epoch+1}: Train Acc={train_accs[-1]:.4f} | Val Acc={val_accs[-1]:.4f}")

    torch.save({
        'epoch': epoch,
        'model_state': model.state_dict(),
        'optimizer_state': optimizer.state_dict(),
        'scaler_state': scaler.state_dict(),
        'train_losses': train_losses, 'val_losses': val_losses,
        'train_accs': train_accs, 'val_accs': val_accs,
    }, CHECKPOINT_PATH)

torch.save(model.state_dict(), f"{SAVE_DIR}\\{MODEL_NAME}_wikiart.pth")
with open(f"{SAVE_DIR}\\{MODEL_NAME}_history.json", "w") as f:
    json.dump({'train_losses': train_losses, 'val_losses': val_losses,
               'train_accs': train_accs, 'val_accs': val_accs}, f)
print("Done!")